# Template Example Notebook

This is a template notebook. The first heading should be the title of what notebook is about. For example, if it is a project on neo4j tutorial the heading should be `Project Title`.

- Add description of what the notebook does.
- Point to references, e.g. (neo4j.example.md)
- Add citations.
- Keep the notebook flow clear.
- Comments should be imperative and have a period at the end.
- Your code should be well commented.

The name of this notebook should in the following format:
- if the notebook is exploring `pycaret API`, then it is `pycaret.example.ipynb`

Follow the reference to write notebooks in a clear manner: https://github.com/causify-ai/helpers/blob/master/docs/coding/all.jupyter_notebook.how_to_guide.md

In [21]:
# %load_ext autoreload
# %autoreload 2
# %matplotlib inline

In [22]:
# import logging
# # Import libraries in this section.
# # Avoid imports like import *, from ... import ..., from ... import *, etc.

# import helpers.hdbg as hdbg
# import helpers.hnotebook as hnotebo

In [23]:
# hdbg.init_logger(verbosity=logging.INFO)

# _LOG = logging.getLogger(__name__)

# hnotebo.config_notebook()

## Make the notebook flow clear
Each notebook needs to follow a clear and logical flow, e.g:
- Load data
- Compute stats
- Clean data
- Compute stats
- Do analysis
- Show results




#############################################################################
Template
#############################################################################

In [24]:
# class Template:
#     """
#     Brief imperative description of what the class does in one line, if needed.
#     """

#     def __init__(self):
#         pass

#     def method1(self, arg1: int) -> None:
#         """
#         Brief imperative description of what the method does in one line.

#         You can elaborate more in the method docstring in this section, for e.g. explaining
#         the formula/algorithm. Every method/function should have a docstring, typehints and include the
#         parameters and return as follows:

#         :param arg1: description of arg1
#         :return: description of return
#         """
#         # Code bloks go here.
#         # Make sure to include comments to explain what the code is doing.
#         # No empty lines between code blocks.
#         pass


# def template_function(arg1: int) -> None:
#     """
#     Brief imperative description of what the function does in one line.

#     You can elaborate more in the function docstring in this section, for e.g. explaining
#     the formula/algorithm. Every function should have a docstring, typehints and include the
#     parameters and return as follows:

#     :param arg1: description of arg1
#     :return: description of return
#     """
#     # Code bloks go here.
#     # Make sure to include comments to explain what the code is doing.
#     # No empty lines between code blocks.
#     pass

## The flow should be highlighted using headings in markdown
```
# Level 1
## Level 2
### Level 3
```

# Description

This notebook implements a full time series forecasting pipeline using the
Darts library to predict S&P 500 index prices and identify which market
sectors will outperform in the near future.

The notebook covers data collection for S&P 500, 11 sector ETFs, and 22
macroeconomic dimensions including yield curve, VIX, CPI, Fed rate, oil,
gold, and dollar index. It performs data preprocessing with release-date-aware
forward fill, feature engineering including technical indicators, calendar
features, and event flags for FOMC meetings, CPI release dates, and US
holidays. It trains the full Darts model suite across baseline, statistical,
probabilistic, and machine learning model groups, applies SHAP based feature
selection, performs hyperparameter tuning, builds ensemble models, and
benchmarks results against Facebook Prophet and Statsmodels. The sector
rotation engine recommends which sectors to rotate into based entirely on
model predictions, historical correlations, and risk adjusted scores.

References:
- Darts documentation: https://unit8co.github.io/darts/
- FRED API documentation: https://fred.stlouisfed.org/docs/api/fred/
- yfinance documentation: https://pypi.org/project/yfinance/
- Prophet documentation: https://facebook.github.io/prophet/

# Imports

In [25]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import logging
import os
import warnings

import darts
import darts.dataprocessing.transformers
import darts.metrics
import darts.models
import darts.timeseries
import dotenv
import fredapi
import holidays
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import prophet
import seaborn as sns
import shap
import sklearn.preprocessing
import statsmodels.tsa.arima.model
import statsmodels.tsa.holtwinters
import tqdm
import utils
import yfinance as yf

warnings.filterwarnings("ignore")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
# Configure the logger to track notebook execution.
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)
_LOG = logging.getLogger(__name__)
# Log the environment setup confirmation and Darts version.
_LOG.info("Environment configured successfully.")
_LOG.info("Darts version: %s", darts.__version__)

2026-04-03 18:03:24,760 - INFO - Environment configured successfully.
2026-04-03 18:03:24,760 - INFO - Darts version: 0.43.0


# Configuration

In [27]:
# Define the date range for all data downloads.
START_DATE = "2018-01-01"
END_DATE = "2024-12-31"
# Define the S&P 500 ticker symbol.
SP500_TICKER = "^GSPC"
# Define all 11 sector ETF ticker symbols.
SECTOR_TICKERS = [
    "XLK",
    "XLV",
    "XLF",
    "XLE",
    "XLY",
    "XLP",
    "XLI",
    "XLU",
    "XLB",
    "XLRE",
    "XLC",
]
# Define sector names mapped to their ticker symbols.
SECTOR_NAMES = {
    "XLK": "Technology",
    "XLV": "Healthcare",
    "XLF": "Financials",
    "XLE": "Energy",
    "XLY": "Consumer Discretionary",
    "XLP": "Consumer Staples",
    "XLI": "Industrials",
    "XLU": "Utilities",
    "XLB": "Materials",
    "XLRE": "Real Estate",
    "XLC": "Communication Services",
}
# Define daily macro indicator ticker symbols from yfinance.
DAILY_MACRO_TICKERS = {
    "VIX": "^VIX",
    "TNX": "^TNX",
    "IRX": "^IRX",
    "OIL": "CL=F",
    "GOLD": "GC=F",
    "DXY": "DX-Y.NYB",
}
# Define monthly macro indicator codes from FRED API.
MONTHLY_MACRO_CODES = {
    "CPI": "CPIAUCSL",
    "CORE_CPI": "CPILFESL",
    "FED_RATE": "FEDFUNDS",
    "UNEMPLOYMENT": "UNRATE",
    "NFP": "PAYEMS",
    "RETAIL_SALES": "RSAFS",
    "INDUSTRIAL_PROD": "INDPRO",
    "PCE": "PCEPI",
    "PPI": "PPIACO",
}
# Define the FRED daily indicator code for breakeven inflation.
BREAKEVEN_INFLATION_CODE = "T10YIE"
# Define the data directory path for saving raw CSV files.
DATA_DIR = "data"
# Define the test set size in trading days.
TEST_SIZE = 60
# Define the validation set size in trading days.
VAL_SIZE = 60
# Define the forecast horizon in trading days.
FORECAST_HORIZON = 30
# Create the data directory if it does not exist.
os.makedirs(DATA_DIR, exist_ok=True)
_LOG.info("Configuration loaded successfully.")
_LOG.info("Date range: %s to %s", START_DATE, END_DATE)
_LOG.info("Sectors: %s", list(SECTOR_NAMES.values()))

2026-04-03 18:03:24,777 - INFO - Configuration loaded successfully.
2026-04-03 18:03:24,777 - INFO - Date range: 2018-01-01 to 2024-12-31
2026-04-03 18:03:24,777 - INFO - Sectors: ['Technology', 'Healthcare', 'Financials', 'Energy', 'Consumer Discretionary', 'Consumer Staples', 'Industrials', 'Utilities', 'Materials', 'Real Estate', 'Communication Services']


# Data collection

This section downloads all required data from Yahoo Finance and the FRED
API and saves it as raw CSV files in the `data` directory. The data
includes S&P 500 historical prices, 11 sector ETF prices, 6 daily
macroeconomic indicators, and 10 monthly macroeconomic indicators covering
the period from 2018 to 2024.

In [28]:
# Load environment variables from the .env file.
dotenv.load_dotenv()
# Read the FRED API key from the environment variables.
FRED_API_KEY = os.environ.get("FRED_API_KEY")
# Verify the FRED API key was loaded successfully.
if FRED_API_KEY is None:
    raise ValueError("FRED_API_KEY not found in .env file.")
_LOG.info("FRED API key loaded successfully.")

2026-04-03 18:03:24,794 - INFO - FRED API key loaded successfully.


In [29]:
# Download S&P 500 historical price data.
sp500 = utils.download_sp500(SP500_TICKER, START_DATE, END_DATE)
sp500.head(5)

2026-04-03 18:03:24,809 - INFO - Downloading S&P 500 data from 2018-01-01 to 2024-12-31.
2026-04-03 18:03:24,827 - INFO - Downloaded 1760 rows of S&P 500 data.


Price,Close,High,Low,Open,Volume
Date,,,,,
2018-01-02,2695.810059,2695.889893,2682.360107,2683.729980,3397430000
2018-01-03,2713.060059,2714.370117,2697.770020,2697.850098,3544030000
2018-01-04,2723.989990,2729.290039,2719.070068,2719.310059,3697340000
2018-01-05,2743.149902,2743.449951,2727.919922,2731.330078,3239280000
2018-01-08,2747.709961,2748.510010,2737.600098,2742.669922,3246160000


In [30]:
# Save the raw S&P 500 data to CSV for reproducibility.
utils.save_data(sp500, "sp500_raw.csv", DATA_DIR)

2026-04-03 18:03:24,847 - INFO - Saved 1760 rows to data/sp500_raw.csv.


In [31]:
# Download historical price data for all 11 sector ETFs.
sectors = utils.download_sectors(SECTOR_TICKERS, START_DATE, END_DATE)
sectors.head(5)

2026-04-03 18:03:24,861 - INFO - Downloading 11 sector ETFs.
2026-04-03 18:03:25,009 - INFO - Downloaded 1760 rows for 11 sectors.


,XLK,XLV,XLF,XLE,XLY,XLP,XLI,XLU,XLB,XLRE,XLC
Date,,,,,,,,,,,
2018-01-02,29.872929,72.723343,23.884140,25.747257,46.275227,45.314564,66.254837,20.132587,25.995674,24.837584,NaN
2018-01-03,30.122103,73.419174,24.012465,26.132860,46.487701,45.298534,66.611694,19.974422,26.177759,24.845160,NaN
2018-01-04,30.274364,73.523544,24.234877,26.290604,46.640125,45.426777,67.099129,19.808548,26.406425,24.420465,NaN
2018-01-05,30.592756,74.149780,24.303310,26.280087,47.009624,45.627136,67.560440,19.800835,26.618153,24.473553,NaN
2018-01-08,30.708111,73.880157,24.269094,26.437824,47.065044,45.739338,67.838966,19.985994,26.656265,24.640400,NaN


In [32]:
# Save the raw sector ETF data to CSV for reproducibility.
utils.save_data(sectors, "sectors_raw.csv", DATA_DIR)

2026-04-03 18:03:25,034 - INFO - Saved 1760 rows to data/sectors_raw.csv.


In [33]:
# Download daily macroeconomic indicators from Yahoo Finance.
macro_daily = utils.download_daily_macro(
    DAILY_MACRO_TICKERS, START_DATE, END_DATE
)
macro_daily.head(5)

2026-04-03 18:03:25,047 - INFO - Downloading 6 daily macro indicators.
2026-04-03 18:03:25,131 - INFO - Downloaded 1760 rows for 6 daily macro indicators.


,VIX,TNX,IRX,OIL,GOLD,DXY
Date,,,,,,
2018-01-02,9.77,2.465,1.378,60.369999,1313.699951,91.849998
2018-01-03,9.15,2.447,1.370,61.630001,1316.199951,92.160004
2018-01-04,9.22,2.453,1.370,62.009998,1319.400024,91.849998
2018-01-05,9.22,2.476,1.370,61.439999,1320.300049,91.949997
2018-01-08,9.52,2.480,1.380,61.730000,1318.599976,92.330002


In [34]:
# Save the raw daily macro indicator data to CSV for reproducibility.
utils.save_data(macro_daily, "macro_daily_raw.csv", DATA_DIR)

2026-04-03 18:03:25,151 - INFO - Saved 1760 rows to data/macro_daily_raw.csv.


In [44]:
# Download monthly macroeconomic indicators from the FRED API.
macro_monthly = utils.download_monthly_macro(
    MONTHLY_MACRO_CODES,
    BREAKEVEN_INFLATION_CODE,
    START_DATE,
    END_DATE,
    FRED_API_KEY,
)
macro_monthly.head(5)

2026-04-04 13:52:28,016 - INFO - Downloading 9 monthly macro indicators from FRED.
2026-04-04 13:52:32,700 - INFO - Downloading 10Y breakeven inflation from FRED.
2026-04-04 13:52:33,212 - INFO - Downloaded 1850 rows for 10 monthly macro indicators.


,CPI,CORE_CPI,FED_RATE,UNEMPLOYMENT,NFP,RETAIL_SALES,INDUSTRIAL_PROD,PCE,PPI,BREAKEVEN
Date,,,,,,,,,,
2018-01-01,248.859,255.204,1.41,4.0,147660.0,481414.0,101.4625,101.199,197.9,NaN
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.00
2018-01-03,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.98
2018-01-04,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.01
2018-01-05,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.01


In [45]:
# Save the raw monthly macro indicator data to CSV for reproducibility.
utils.save_data(macro_monthly, "macro_monthly_raw.csv", DATA_DIR)

2026-04-04 13:54:08,387 - INFO - Saved 1850 rows to data/macro_monthly_raw.csv.


# Data preprocessing and EDA

This section cleans and aligns all downloaded datasets, applies
release-date-aware forward fill to monthly macro indicators, adds
binary release flags to distinguish actual data release days from
forward filled values, and performs exploratory data analysis to
understand the structure and patterns in the data before modeling.

## Align datasets to common date index

In [63]:
# Use S&P 500 date index as the master index for all datasets.
master_index = sp500.index
# Reindex daily datasets directly — no monthly values to preserve.
sectors = sectors.reindex(master_index)
macro_daily = macro_daily.reindex(master_index)
# Preserve monthly values before reindexing to avoid losing data
# that falls on non trading days like weekends and holidays.
macro_monthly = utils.preserve_month_start_values(
    macro_monthly, master_index
)
# Verify all datasets have the same shape after alignment.
_LOG.info("S&P 500 shape: %s", sp500.shape)
_LOG.info("Sectors shape: %s", sectors.shape)
_LOG.info("Macro daily shape: %s", macro_daily.shape)
_LOG.info("Macro monthly shape: %s", macro_monthly.shape)

2026-04-04 16:08:45,975 - INFO - S&P 500 shape: (1742, 5)
2026-04-04 16:08:45,975 - INFO - Sectors shape: (1742, 11)
2026-04-04 16:08:45,975 - INFO - Macro daily shape: (1742, 6)
2026-04-04 16:08:45,975 - INFO - Macro monthly shape: (1742, 19)


## Apply release-date-aware forward fill to monthly indicators

In [64]:
# Apply release-date-aware forward fill to monthly macro indicators.
macro_monthly = utils.apply_release_aware_forward_fill(
    macro_monthly,
    FRED_API_KEY,
    MONTHLY_MACRO_CODES,
)
# Preview the result to verify forward fill worked correctly.
macro_monthly.head(20)

2026-04-04 16:08:45,992 - INFO - Applying release-date-aware forward fill using vintage dates.
Forward filling indicators: 100%|█████████████████| 9/9 [00:03<00:00,  2.69it/s]
2026-04-04 16:08:49,335 - INFO - Forward fill complete for 9 indicators.


,CPI,CORE_CPI,FED_RATE,UNEMPLOYMENT,NFP,RETAIL_SALES,INDUSTRIAL_PROD,PCE,PPI,BREAKEVEN,CPI_released,CORE_CPI_released,FED_RATE_released,UNEMPLOYMENT_released,NFP_released,RETAIL_SALES_released,INDUSTRIAL_PROD_released,PCE_released,PPI_released
Date,,,,,,,,,,,,,,,,,,,
2018-01-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.199,NaN,2.09,0,0,0,0,0,0,0,1,0
2018-01-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.199,NaN,2.10,0,0,0,0,0,0,0,0,0
2018-01-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,101.199,NaN,2.11,0,0,0,0,0,0,0,0,0
2018-02-01,NaN,NaN,1.42,NaN,NaN,NaN,NaN,101.199,NaN,2.11,0,0,1,0,0,0,0,0,0
2018-02-02,NaN,NaN,1.42,4.1,148054.0,NaN,NaN,101.199,NaN,2.14,0,0,0,1,1,0,0,0,0
2018-02-05,NaN,NaN,1.42,4.1,148054.0,NaN,NaN,101.199,NaN,2.10,0,0,0,0,0,0,0,0,0
2018-02-06,NaN,NaN,1.42,4.1,148054.0,NaN,NaN,101.199,NaN,2.10,0,0,0,0,0,0,0,0,0
2018-02-07,NaN,NaN,1.42,4.1,148054.0,NaN,NaN,101.199,NaN,2.10,0,0,0,0,0,0,0,0,0
2018-02-08,NaN,NaN,1.42,4.1,148054.0,NaN,NaN,101.199,NaN,2.09,0,0,0,0,0,0,0,0,0


In [65]:
# Check when the first non-NaN values appear for each indicator.
first_valid = macro_monthly.apply(lambda col: col.first_valid_index())
_LOG.info("First valid date per indicator:\n%s", first_valid.to_string())

2026-04-04 16:08:49,371 - INFO - First valid date per indicator:
CPI                        2018-02-14
CORE_CPI                   2018-02-14
FED_RATE                   2018-02-01
UNEMPLOYMENT               2018-02-02
NFP                        2018-02-02
RETAIL_SALES               2018-02-14
INDUSTRIAL_PROD            2018-02-15
PCE                        2018-01-29
PPI                        2018-02-15
BREAKEVEN                  2018-01-29
CPI_released               2018-01-29
CORE_CPI_released          2018-01-29
FED_RATE_released          2018-01-29
UNEMPLOYMENT_released      2018-01-29
NFP_released               2018-01-29
RETAIL_SALES_released      2018-01-29
INDUSTRIAL_PROD_released   2018-01-29
PCE_released               2018-01-29
PPI_released               2018-01-29


In [66]:
# Verify the forward fill and release flags worked correctly.
_LOG.info("Macro monthly shape: %s", macro_monthly.shape)
_LOG.info("Total NaN values remaining: %d", macro_monthly.isnull().sum().sum())
_LOG.info("Release flag columns: %s", 
    [col for col in macro_monthly.columns if col.endswith("_released")]
)
# Show a sample around a known release date to verify flags.
macro_monthly.loc["2018-01-01":"2018-02-05", ["CPI", "CPI_released"]].head(10)

2026-04-04 16:08:49,390 - INFO - Macro monthly shape: (1742, 19)
2026-04-04 16:08:49,391 - INFO - Total NaN values remaining: 73
2026-04-04 16:08:49,391 - INFO - Release flag columns: ['CPI_released', 'CORE_CPI_released', 'FED_RATE_released', 'UNEMPLOYMENT_released', 'NFP_released', 'RETAIL_SALES_released', 'INDUSTRIAL_PROD_released', 'PCE_released', 'PPI_released']


,CPI,CPI_released
Date,,
2018-01-29,NaN,0
2018-01-30,NaN,0
2018-01-31,NaN,0
2018-02-01,NaN,0
2018-02-02,NaN,0
2018-02-05,NaN,0


In [67]:
# Verify CPI release flag shows 1 on actual release days.
cpi_releases = macro_monthly[macro_monthly["CPI_released"] == 1]["CPI"]
_LOG.info(
    "Number of actual CPI release days: %d", len(cpi_releases)
)
_LOG.info(
    "First few CPI release dates:\n%s", cpi_releases.head(5).to_string()
)

2026-04-04 16:08:49,415 - INFO - Number of actual CPI release days: 89
2026-04-04 16:08:49,415 - INFO - First few CPI release dates:
Date
2018-02-14    249.529
2018-03-13    249.577
2018-04-11    250.227
2018-05-10    250.792
2018-06-12    251.018


In [68]:
# Check fredapi version and available methods.
import fredapi
_LOG.info("fredapi version: %s", fredapi.__version__)
fred_temp = fredapi.Fred(api_key=FRED_API_KEY)
fred_methods = [m for m in dir(fred_temp) if not m.startswith("_")]
_LOG.info("Available methods: %s", fred_methods)

2026-04-04 16:08:49,433 - INFO - fredapi version: 0.5.2
2026-04-04 16:08:49,433 - INFO - Available methods: ['api_key', 'earliest_realtime_start', 'get_series', 'get_series_all_releases', 'get_series_as_of_date', 'get_series_first_release', 'get_series_info', 'get_series_latest_release', 'get_series_vintage_dates', 'latest_realtime_end', 'max_results_per_request', 'nan_char', 'proxies', 'root_url', 'search', 'search_by_category', 'search_by_release']


In [69]:
# Verify CPI release flag shows 1 on true release days.
cpi_releases = macro_monthly[macro_monthly["CPI_released"] == 1]["CPI"]
_LOG.info("Number of actual CPI release days: %d", len(cpi_releases))
_LOG.info(
    "First few CPI release dates:\n%s", cpi_releases.head(5).to_string()
)
# Verify total NaN values remaining.
_LOG.info(
    "Total NaN values remaining: %d",
    macro_monthly.isnull().sum().sum()
)

2026-04-04 16:08:49,450 - INFO - Number of actual CPI release days: 89
2026-04-04 16:08:49,451 - INFO - First few CPI release dates:
Date
2018-02-14    249.529
2018-03-13    249.577
2018-04-11    250.227
2018-05-10    250.792
2018-06-12    251.018
2026-04-04 16:08:49,452 - INFO - Total NaN values remaining: 73


In [70]:
# Identify which columns still have NaN values and how many.
nan_counts = macro_monthly.isnull().sum()
nan_counts = nan_counts[nan_counts > 0]
_LOG.info("Columns with remaining NaN values:\n%s", nan_counts.to_string())

2026-04-04 16:08:49,469 - INFO - Columns with remaining NaN values:
CPI                12
CORE_CPI           12
FED_RATE            3
UNEMPLOYMENT        4
NFP                 4
RETAIL_SALES       12
INDUSTRIAL_PROD    13
PPI                13


In [71]:
# Find the first date where all monthly macro indicators have values.
first_complete_date = macro_monthly.dropna().index[0]
_LOG.info(
    "First date with complete macro data: %s", first_complete_date
)
# Count how many rows we lose by starting from this date.
rows_before = len(macro_monthly)
rows_after = len(macro_monthly.loc[first_complete_date:])
rows_dropped = rows_before - rows_after
_LOG.info(
    "Rows dropped: %d (%.2f%% of total)",
    rows_dropped,
    rows_dropped / rows_before * 100,
)

2026-04-04 16:08:49,485 - INFO - First date with complete macro data: 2018-02-15 00:00:00
2026-04-04 16:08:49,486 - INFO - Rows dropped: 13 (0.75% of total)


## Trim datasets to first complete macro data date

In [72]:
# Trim all datasets to start from the first date where all macro
# indicators have complete values to eliminate initial NaN period.
sp500 = sp500.loc[first_complete_date:]
sectors = sectors.loc[first_complete_date:]
macro_daily = macro_daily.loc[first_complete_date:]
macro_monthly = macro_monthly.loc[first_complete_date:]
# Show percentage of rows dropped across all datasets.
_LOG.info(
    "Rows dropped: %d (%.2f%% of total)",
    rows_dropped,
    rows_dropped / rows_before * 100,
)
# Verify all datasets have the same shape after trimming.
_LOG.info("S&P 500 shape after trim: %s", sp500.shape)
_LOG.info("Sectors shape after trim: %s", sectors.shape)
_LOG.info("Macro daily shape after trim: %s", macro_daily.shape)
_LOG.info("Macro monthly shape after trim: %s", macro_monthly.shape)
# Verify no NaN values remain in monthly macro data.
_LOG.info(
    "NaN values remaining in macro monthly: %d",
    macro_monthly.isnull().sum().sum()
)

2026-04-04 16:08:49,502 - INFO - Rows dropped: 13 (0.75% of total)
2026-04-04 16:08:49,502 - INFO - S&P 500 shape after trim: (1729, 5)
2026-04-04 16:08:49,502 - INFO - Sectors shape after trim: (1729, 11)
2026-04-04 16:08:49,502 - INFO - Macro daily shape after trim: (1729, 6)
2026-04-04 16:08:49,503 - INFO - Macro monthly shape after trim: (1729, 19)
2026-04-04 16:08:49,503 - INFO - NaN values remaining in macro monthly: 0


## Reconstruct XLC pre-launch history

XLC (Communication Services ETF) launched on June 19 2018. To preserve
the full analysis period from January 2018, pre-launch XLC values are
reconstructed using historical prices of the top 12 XLC constituent
companies weighted by market capitalization at each point in time.

The reconstruction is validated against actual XLC prices for the
overlap period from June 2018 onwards. If the correlation exceeds
0.90 the reconstruction replaces NaN values in the sectors DataFrame.
If correlation falls below 0.90 the dataset is trimmed to June 2018
as a fallback — ensuring we never use an unreliable reconstruction.

References:
- XLC constituent data: https://www.ssga.com/us/en/intermediary/etfs/funds/the-communication-services-select-sector-spdr-fund-xlc

In [73]:
# Reconstruct XLC pre-launch history using verified constituent weights
# from the official SPDR ETF filing on June 19 2018.
sectors = utils.reconstruct_xlc(
    sectors,
    START_DATE,
    str(xlc_first_date.date()),
    correlation_threshold=0.90,
)
# Verify XLC NaN values are resolved after reconstruction.
xlc_nan_remaining = sectors["XLC"].isnull().sum()
_LOG.info(
    "XLC NaN values remaining after reconstruction: %d",
    xlc_nan_remaining,
)
# Show percentage of rows that were reconstructed.
_LOG.info(
    "XLC rows reconstructed: %d (%.2f%% of total)",
    98 - xlc_nan_remaining,
    (98 - xlc_nan_remaining) / len(sectors) * 100,
)
# Preview XLC values around the reconstruction period.
sectors[["XLC"]].loc[
    sectors.index[sectors.index < pd.Timestamp(xlc_first_date)]
].tail(10)

2026-04-04 16:08:49,519 - INFO - Downloading XLC constituent stock prices.
2026-04-04 16:08:49,547 - INFO - Downloaded GOOGL successfully.
2026-04-04 16:08:49,562 - INFO - Downloaded GOOG successfully.
2026-04-04 16:08:49,575 - INFO - Downloaded DIS successfully.
2026-04-04 16:08:49,589 - INFO - Downloaded CMCSA successfully.
2026-04-04 16:08:49,604 - INFO - Downloaded NFLX successfully.
2026-04-04 16:08:49,618 - INFO - Downloaded T successfully.
2026-04-04 16:08:49,632 - INFO - Downloaded VZ successfully.
2026-04-04 16:08:49,658 - INFO - Downloaded EA successfully.
2026-04-04 16:08:49,671 - INFO - Downloaded TTWO successfully.
2026-04-04 16:08:49,685 - INFO - Downloaded OMC successfully.
2026-04-04 16:08:49,699 - INFO - Downloaded TMUS successfully.
2026-04-04 16:08:49,713 - INFO - Downloaded TRIP successfully.
2026-04-04 16:08:49,726 - INFO - Downloaded NWSA successfully.
2026-04-04 16:08:49,740 - INFO - Downloaded LUMN successfully.
2026-04-04 16:08:49,741 - INFO - Using 16 constitu

,XLC
Date,
2018-06-05,44.523964
2018-06-06,44.625136
2018-06-07,44.287402
2018-06-08,44.416390
2018-06-11,44.915986
2018-06-12,45.284413
2018-06-13,45.258189
2018-06-14,46.302524
2018-06-15,46.281115


In [74]:
# Verify the full sectors DataFrame has zero NaN values.
_LOG.info(
    "Total NaN values in sectors: %d",
    sectors.isnull().sum().sum()
)
# Verify all sectors start from the same date.
_LOG.info("Sectors date range: %s to %s",
    sectors.index[0].date(),
    sectors.index[-1].date()
)
# Preview sectors DataFrame.
sectors.head(10)

2026-04-04 16:08:49,763 - INFO - Total NaN values in sectors: 0
2026-04-04 16:08:49,764 - INFO - Sectors date range: 2018-02-15 to 2024-12-30


,XLK,XLV,XLF,XLE,XLY,XLP,XLI,XLU,XLB,XLRE,XLC
Date,,,,,,,,,,,
2018-02-15,31.044949,73.758400,24.825132,23.938467,48.635406,44.384865,67.255798,19.164339,25.703501,23.116014,44.850350
2018-02-16,30.998814,74.288979,24.790913,23.878881,48.399853,44.561199,67.273209,19.334070,25.610334,23.267694,44.673643
2018-02-20,31.026497,73.471359,24.713930,23.749174,48.252060,43.551361,66.594315,19.083328,25.496004,23.032589,44.381223
2018-02-21,30.828087,73.166939,24.748146,23.349558,48.182777,43.030403,66.611694,18.832590,25.407080,22.607887,44.613615
2018-02-22,30.864992,73.053856,24.542837,23.605453,48.242825,43.142609,67.055611,18.929028,25.576460,22.865746,44.588582
2018-02-23,31.510994,74.219398,24.910683,24.117247,48.917160,43.559368,67.612686,19.422792,25.953331,23.282862,45.553768
2018-02-26,32.004723,75.158775,25.287077,24.260971,49.328220,43.831860,68.543991,19.364933,26.067669,23.366285,45.864453
2018-02-27,31.681719,74.375954,25.047550,23.945478,48.284397,43.254814,67.682304,19.056326,25.707731,22.865746,44.836946
2018-02-28,31.455639,73.184334,24.696819,23.395130,48.067314,42.805984,66.716141,18.925171,25.250404,22.835411,44.107506
